# Notebook 1

This notebook is to load our data and clean our dataset
The cleaned dataset should include:

*   Pregnancies with 2 EHG recordings (pre-filtered) - Chelsea

-- Also made the total number of weeks between variable
-- Save the cleaned dataset as a new zip file and downloaded it onto git hub



NEW NOTEBOOK:

* Jerold Calculate Signal 4? - Used both filtered & raw signals for the calculation
* All EHG recordings have 150 seconds trimmed on either side. Create new variables for trimmed data - Michael
* All EHG recordings will be visually checked for signal quality - Chelsea Delegate
* Seperate tables will be made for each signal type, with matched EHG ids, signal type, pregancy id, and outcome - Chelsea




## Step One: Load the dataset from PhysioNet

In [ ]:
!pip install wfdb

In [ ]:
# Libraries that we will need either now or later
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
from IPython.display import display

In [ ]:
!wget -r -N -c -np -nH --cut-dirs=3 \
https://physionet.org/files/icehg-ds/1.0.1/

--2026-10-26 19:46:00--  https://physionet.org/files/icehg-ds/1.0.1/
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘index.html’

index.html              [ <=>                ]   2.43K  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-10-26 19:46:01 (432 MB/s) - ‘index.html’ saved [2484]

Loading robots.txt; please ignore errors.
--2026-10-26 19:46:01--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK

    The file is already fully retrieved; nothing to do.

--2026-10-26 19:46:02--  https://physionet.org/files/icehg-ds/1.0.1/early_cesarean/
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘early_cesarean/index

# Step 2: Locate the PhysioNet download and the three metadata tables

In [ ]:
# Set the path to the downloaded PhysioNet dataset
data_path = os.getcwd()

# Verify that all three clinical metadata tables are available
for filename in ["Table_1.txt", "Table_2.txt", "Table_3.txt"]:
    file_path = os.path.join(data_path, filename)

    # Stop execution if a required metadata file is missing
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"Missing metadata file: {filename}")

# Confirm the dataset was located successfully
print("ICEHG dataset located at:", data_path)
print("All three metadata tables found!")

ICEHG dataset located at: /content
All three metadata tables found!


## Step 3: Import and combine tables 1-3 so that they make a pandas dataframe.

In [ ]:
# Define the 12 clinical metadata columns
columns = [
    "pregnancy_id", "age", "early_id", "early_week",
    "weight_early", "later_id", "later_week",
    "weight_later", "height", "placenta",
    "gestation", "newborn_weight"
]

# Define a function to import each clinical metadata table and process them so it can be converted into a pandas dataframe
def load_metadata(filename, outcome):

    # Create an empty list to store pregnancy records
    rows = []

    # Open the metadata file and read its contents
    with open(os.path.join(data_path, filename),
              "r", encoding="utf-8-sig") as file:

        # Read each line in the text file
        for line in file:

            # Skip table borders and other non-data lines
            if not line.startswith("|"):
                continue

            # Separate column values and remove extra whitespace
            values = [
                value.strip()
                for value in line.strip().strip("|").split("|")
            ]

            # Retain only rows beginning with a pregnancy ID
            if values and values[0].isdigit():

                # Verify that each row contains all 12 columns
                if len(values) != len(columns):
                    raise ValueError(
                        f"Unexpected columns in {filename}"
                    )

                # Add the pregnancy record to the list
                rows.append(values)

    # Convert the extracted records into a pandas DataFrame
    result = pd.DataFrame(rows, columns=columns)

    # Identify the delivery outcome associated with each pregnancy
    result["outcome"] = outcome

    # Return the completed DataFrame
    return result


# Import and vertically combine the three delivery outcome groups
df = pd.concat([
    load_metadata("Table_1.txt", "induced"),
    load_metadata("Table_2.txt", "cesarean"),
    load_metadata("Table_3.txt", "induced-cesarean")
], ignore_index=True)

# Display the number of pregnancies before matching
print("Total pregnancies:", len(df))

# Display the number of pregnancies in each delivery outcome group
print(df["outcome"].value_counts())

# Preview the combined clinical metadata
display(df.head())


Total pregnancies: 91
outcome
induced             59
induced-cesarean    19
cesarean            13
Name: count, dtype: int64


,pregnancy_id,age,early_id,early_week,weight_early,later_id,later_week,weight_later,height,placenta,gestation,newborn_weight,outcome
0,1,34,691,22.9,65,~,~,~,165,front,38.1,3330,induced
1,2,33,713,23.1,108,~,~,~,164,front,39.7,4250,induced
2,3,32,1064,22.7,60,~,~,~,160,front,41.1,3750,induced
3,4,39,1077,22.3,70,~,~,~,163,front,39.0,3330,induced
4,5,35,1086,22.1,67,~,~,~,165,end,38.6,2970,induced


# Step 3: Identify Matched Preganancies and Validate the metadata

This step filters that dataset to retain only pregnancies with both an early and later recording, converts the metadata into the correct data types, and checks that the remaining data are suitable for your analysis.

In [ ]:
# Standardize missing values
# PhysioNet uses "~" to indicate missing information.
# Replace this symbol with pandas' recognized missing-value marker.
df = df.replace("~", pd.NA)

# Identify matched pregnancies
# Retain only pregnancies with both an early and later recording ID.
# These pregnancies can be used to calculate within-pregnancy EHG changes.
# .copy() creates a separate DataFrame without modifying the original.
df_matched = df.dropna(subset=["early_id", "later_id"]).copy()

# Convert numerical metadata into appropriate data types
# Values imported from the text tables are initially stored as strings.
# Convert numerical columns so they can be used in calculations.
numeric_columns = [
    "pregnancy_id", "age", "early_id", "early_week",
    "weight_early", "later_id", "later_week",
    "weight_later", "height", "gestation", "newborn_weight"
]

for column in numeric_columns:
    df_matched[column] = pd.to_numeric(
        df_matched[column], errors="raise"
    )

# Calculate the gestational age difference between recordings
# This will later allow changes in EHG features (e.g., median frequency)
# to be normalized by the number of weeks between recording sessions.
df_matched["weeks_between"] = (
    df_matched["later_week"] - df_matched["early_week"]
)

# Validate the matched pregnancy dataset
# Check that each pregnancy appears only once.
if df_matched["pregnancy_id"].duplicated().any():
    raise ValueError("Duplicate pregnancy IDs detected.")

# Check that gestational ages are available for both recordings.
if df_matched[["early_week", "later_week", "weeks_between"]].isna().any().any():
    raise ValueError("Missing gestational ages in matched pregnancies.")

# Check that the later recording occurred after the early recording.
if not (df_matched["weeks_between"] > 0).all():
    raise ValueError("Invalid recording chronology detected.")

# Reset the DataFrame index
# Renumber the rows consecutively without changing the pregnancy IDs.
df_matched = df_matched.reset_index(drop=True)

# Summarize the matched dataset
print("Total matched pregnancies:", len(df_matched))
print("Total required EHG recordings:", 2 * len(df_matched))

# Display the number of matched pregnancies in each delivery outcome group.
print("\nMatched pregnancies by delivery outcome:")
print(df_matched["outcome"].value_counts())

# Display descriptive statistics for the time between recordings.
print("\nWeeks between recordings:")
print(df_matched["weeks_between"].describe())

# Display the complete matched pregnancy metadata table in Colab.
display(df_matched)


Total matched pregnancies: 35
Total required EHG recordings: 70

Matched pregnancies by delivery outcome:
outcome
induced             22
induced-cesarean     7
cesarean             6
Name: count, dtype: int64

Weeks between recordings:
count    35.000000
mean      7.991429
std       0.913742
min       5.700000
25%       7.550000
50%       7.900000
75%       8.450000
max      11.000000
Name: weeks_between, dtype: float64


,pregnancy_id,age,early_id,early_week,weight_early,later_id,later_week,weight_later,height,placenta,gestation,newborn_weight,outcome,weeks_between
0,17,22,620,23.9,73,1529,31.9,81,168,end,41.1,3930,induced,8.0
1,18,38,656,22.4,77,1385,31.1,82,167,end,41.3,3710,induced,8.7
2,19,26,665,23.1,83,664,30.4,84,165,front,37.1,2860,induced,7.3
3,20,32,757,22.4,68,756,30.6,72,162,end,39.7,3500,induced,8.2
4,21,29,774,25.0,77,775,31.1,83,171,front,41.6,3180,induced,6.1
5,22,32,806,22.4,85,805,30.1,89,158,end,41.1,3700,induced,7.7
6,23,31,907,21.4,78,908,30.4,80,163,end,38.6,2360,induced,9.0
7,24,28,958,22.3,79,956,30.3,82,173,end,40.0,3350,induced,8.0
8,25,32,1011,22.4,70,1010,30.6,75,175,front,41.0,4380,induced,8.2
9,26,35,1081,23.1,69,1079,30.9,72,157,front,37.0,2810,induced,7.8


# Step 4: Verify and copy the corresponding raw EHG recordings
This step uses the 35 matched pregnancies identified in Step 3 to locate their corresponding EHG files. It checks that every recording has both a .dat signal file and a .hea header file before copying them into a new folder.

In [ ]:

# Objective: Extract only the raw EHG recordings corresponding to the
# 35 matched pregnancies, preserving both early and later measurements.
# This step does NOT filter, crop, or otherwise modify the EHG signals.


# Convert the dataset location into a Path object for easier file navigation.
data_path = Path(data_path)

# Define the output folder for the matched dataset.
output_folder = Path("/content/BMEN600_Matched_EHG_Dataset")

# Define the subfolder where matched recordings will be stored.
recordings_output = output_folder / "recordings"

# Remove any recordings copied during previous runs.
# This prevents unmatched or outdated files from remaining in the output.
if recordings_output.exists():
    shutil.rmtree(recordings_output)

# Identify the required EHG files
# Create lists to track files available for copying and missing files.
copy_plan = []
missing = []

# Loop through each pregnancy in the matched metadata table.
for _, pregnancy in df_matched.iterrows():

    # Each pregnancy should have an early and a later EHG recording.
    for stage in ("early", "later"):

        # Retrieve the recording ID for the current measurement session.
        record_id = int(pregnancy[f"{stage}_id"])

        # Identify the PhysioNet folder using the recording stage
        # and delivery outcome (e.g., early_induced).
        folder_name = f"{stage}_{pregnancy['outcome']}"

        # Define the original and destination folders.
        source_folder = data_path / folder_name
        destination_folder = recordings_output / folder_name

        # Each EHG recording requires two files:
        # .dat contains the raw signal data.
        # .hea contains recording information needed to read the signal.
        for extension in (".dat", ".hea"):

            # Construct the filename using the recording ID.
            filename = f"icehg{record_id}{extension}"
            source = source_folder / filename
            destination = destination_folder / filename

            # Record the file for copying if it exists.
            if source.is_file():
                copy_plan.append((source, destination))

            # Otherwise, record the missing file.
            else:
                missing.append(str(source))

# Verify that all required files are available
# Each pregnancy requires:
# 2 recording sessions (early and later) x 2 files (.dat and .hea).
expected_files = len(df_matched) * 2 * 2

# Stop execution if any required file is missing.
# This prevents the creation of an incomplete matched dataset.
if missing or len(copy_plan) != expected_files:
    raise FileNotFoundError(
        f"Expected {expected_files} files, but found {len(copy_plan)}. "
        f"Missing files: {missing}"
    )

# Copy the verified files into the matched dataset folder
# Preserve the original early/later and delivery outcome organization.
for source, destination in copy_plan:

    # Create the destination folder if it does not already exist.
    destination.parent.mkdir(parents=True, exist_ok=True)

    # Copy the original file without changing its contents.
    shutil.copy2(source, destination)

# Verify the completed dataset
# Count the copied signal files and recording headers.
copied_dat = list(recordings_output.rglob("*.dat"))
copied_hea = list(recordings_output.rglob("*.hea"))

# Confirm that all expected recordings were copied successfully.
assert len(copied_dat) == len(copied_hea) == 2 * len(df_matched)

# Display a summary of the completed file selection.
print("Matched pregnancies:", len(df_matched))
print("Raw EHG recordings:", len(copied_dat))
print("Signal files (.dat):", len(copied_dat))
print("Header files (.hea):", len(copied_hea))
print("Missing required files:", len(missing))
print("Output folder:", recordings_output)


Matched pregnancies: 35
Raw EHG recordings: 70
Signal files (.dat): 70
Header files (.hea): 70
Missing required files: 0
Output folder: /content/BMEN600_Matched_EHG_Dataset/recordings


## Step 5: Create and Download the Matched EHG Dataset
Objective: Save the matched pregnancy metadata as a CSV and package
it with the 70 raw EHG recordings into one downloadable ZIP file.

In [ ]:
# Save the matched pregnancy metadata as a CSV file
# This contains pregnancy IDs, recording IDs, gestational ages,
# delivery outcomes, and weeks between recording sessions.

# Ensure the output folder exists.
output_folder.mkdir(parents=True, exist_ok=True)

# Define the CSV filename and save the matched metadata.
csv_path = output_folder / "matched_pregnancies.csv"
df_matched.to_csv(csv_path, index=False)

# Confirm that the CSV was successfully saved.
print("Metadata CSV saved:", csv_path)

# Display the complete matched pregnancy metadata table in Colab.
display(df_matched)


# 5.2 Create a ZIP archive of the matched dataset
# Include the metadata CSV and all 70 matched EHG recordings.
# Each recording has a .dat signal file and a .hea header file.

# The ZIP filename can differ from the name of the existing folder.
zip_path = shutil.make_archive(
    "/content/BMEN600_Matched_Pregnancies_EHG_Dataset",
    "zip",
    root_dir="/content",
    base_dir="BMEN600_Matched_EHG_Dataset"
)



Metadata CSV saved: /content/BMEN600_Matched_EHG_Dataset/matched_pregnancies.csv


,pregnancy_id,age,early_id,early_week,weight_early,later_id,later_week,weight_later,height,placenta,gestation,newborn_weight,outcome,weeks_between
0,17,22,620,23.9,73,1529,31.9,81,168,end,41.1,3930,induced,8.0
1,18,38,656,22.4,77,1385,31.1,82,167,end,41.3,3710,induced,8.7
2,19,26,665,23.1,83,664,30.4,84,165,front,37.1,2860,induced,7.3
3,20,32,757,22.4,68,756,30.6,72,162,end,39.7,3500,induced,8.2
4,21,29,774,25.0,77,775,31.1,83,171,front,41.6,3180,induced,6.1
5,22,32,806,22.4,85,805,30.1,89,158,end,41.1,3700,induced,7.7
6,23,31,907,21.4,78,908,30.4,80,163,end,38.6,2360,induced,9.0
7,24,28,958,22.3,79,956,30.3,82,173,end,40.0,3350,induced,8.0
8,25,32,1011,22.4,70,1010,30.6,75,175,front,41.0,4380,induced,8.2
9,26,35,1081,23.1,69,1079,30.9,72,157,front,37.0,2810,induced,7.8


In [ ]:
# Verify the ZIP archive
# Check that the archive contains the expected files.

import zipfile

with zipfile.ZipFile(zip_path, "r") as archive:
    filenames = archive.namelist()

    # Count the signal, header, and metadata files.
    dat_files = [f for f in filenames if f.endswith(".dat")]
    hea_files = [f for f in filenames if f.endswith(".hea")]
    csv_files = [f for f in filenames if f.endswith(".csv")]

    # Confirm that all required files are included.
    assert len(dat_files) == 70, "Incorrect number of signal files."
    assert len(hea_files) == 70, "Incorrect number of header files."
    assert len(csv_files) == 1, "Incorrect number of metadata CSV files."

# Calculate the ZIP file size in megabytes.
zip_size_mb = Path(zip_path).stat().st_size / 1_000_000

# Display the final dataset summary.
print("\nZIP archive created and verified successfully!")
print("ZIP location:", zip_path)
print(f"ZIP size: {zip_size_mb:.1f} MB")
print("Matched pregnancies:", len(df_matched))
print("EHG recordings included:", len(dat_files))
print("Signal files (.dat):", len(dat_files))
print("Header files (.hea):", len(hea_files))
print("Metadata CSV files:", len(csv_files))


ZIP archive created and verified successfully!
ZIP location: /content/BMEN600_Matched_Pregnancies_EHG_Dataset.zip
ZIP size: 22.6 MB
Matched pregnancies: 35
EHG recordings included: 70
Signal files (.dat): 70
Header files (.hea): 70
Metadata CSV files: 1


In [ ]:
# Download the ZIP file to your computer
# Google Colab will initiate a browser download.

from google.colab import files
files.download(zip_path)

# The ZIP can subsequently be uploaded to the team's GitHub repository,
# subject to PhysioNet's data redistribution requirements.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>